# Drive API handoff — replacement cells after migration Cells1–4
Run these in the SAME existing notebook/kernel, not a new session. Cells1–4 remain completed.
No SSH, tunnel, VPS or reinstall. Keep the T4 session alive. Original source/checkpoint/evidence stay intact.
All executable cells stop before optimizer updates; the training command is commented out.


## Replacement5A — load only the small v6 operational patch
Upload recovery-tools-v6.zip from Linux runs/drive-handoff-prep-v1/ using the chooser below.
Do NOT upload the large017000 ZIP here. No approved source/data/environment setup is rerun.


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import files

assert all(
    name in globals()
    for name in (
        "context",
        "gold",
        "RUN_ID",
        "DRIVE_ROOT",
        "SOURCE",
        "DATASET",
        "STAGING",
        "PYTHON",
        "versions",
        "safe_extract",
        "EXPECTED_SECOND_ACCOUNT_EMAIL",
    )
), "STOP: return to the existing session where migration Cells1-4 passed"
context = globals()["context"]
gold = globals()["gold"]
RUN_ID = globals()["RUN_ID"]
DRIVE_ROOT = globals()["DRIVE_ROOT"]
SOURCE = globals()["SOURCE"]
DATASET = globals()["DATASET"]
STAGING = globals()["STAGING"]
PYTHON = globals()["PYTHON"]
versions = globals()["versions"]
safe_extract = globals()["safe_extract"]
EXPECTED_SECOND_ACCOUNT_EMAIL = globals()["EXPECTED_SECOND_ACCOUNT_EMAIL"]
assert PYTHON.is_file() and SOURCE.is_dir() and DATASET.is_dir()
DRIVE_RUN_ID = RUN_ID + "-drive-v1"  # new, unused; preserve all earlier attempts
DRIVE_HELPERS = Path("/content") / (DRIVE_RUN_ID + "-helpers-v6")
uploaded = files.upload()  # select ONLY recovery-tools-v6.zip (~small metadata/code)
assert set(uploaded) == {"recovery-tools-v6.zip"}
patch_bytes = uploaded["recovery-tools-v6.zip"]
assert hashlib.sha256(patch_bytes).hexdigest() == (
    "cc30fc162fae8f23462c32d71195bd26390e23a08631eb1c3e1a6a8d36a3b47d"
)
PATCH = Path("/content") / (DRIVE_RUN_ID + "-patch.zip")
with PATCH.open("xb") as stream:
    stream.write(patch_bytes)
safe_extract(PATCH, DRIVE_HELPERS)
sys.path.insert(0, str(DRIVE_HELPERS))

print("V6_PATCH_VERIFIED: completed setup reused; SSH unused")

## Replacement5B — authorize API writes, create a NEW second-owned folder and verify a1MiB canary
This is the only additional Google authorization: read existing input and create app-owned output files.
Choose the second account again if prompted. If scopes/creation are denied, STOP; do not skip checks.
The broker refreshes a short-lived bearer token in a0600 local file outside all archives. Never upload it.


In [ ]:
from drive_handoff import DriveAPI
from drive_handoff_setup import setup

CONFIG, folder, BROKER_STOP = setup(context, EXPECTED_SECOND_ACCOUNT_EMAIL, DRIVE_RUN_ID, gold)
config = json.loads(CONFIG.read_text(encoding="utf-8"))
client = DriveAPI(config)
print("API_DESTINATION_READY: second-account-owned, read-back verified")

Expected: new folder owned by the second principal; successful resumable upload and actual API
SHA-256 read-back of the small canary. This is not a full-size upload speed test. Quota must have
about6.1GB free for the new checkpoints/export (more if a Linux input ZIP is uploaded to this account).
No mounted-Drive writes are used. STOP on any auth, ownership, quota, size, checksum or receipt error.

## Replacement6 — stage only bytes matching the Linux017000 backup, then prepare
First try the shared017000, read-only, against EVERY Linux-pinned file hash AND COMPLETE hash.
If unavailable, either provide an existing local Linux-golden ZIP or upload that verified ZIP via
Drive UI while signed into the second account (e.g. into the new folder above), then paste its file ID.
Linux ZIP: runs/durable-recovery-prep-v1/checkpoint-017000.zip, 1,610,503,770 bytes,
SHA e34c9596d0fdaef579b734829d45a60c257370d9de4e17348055de6734366109.
No incomplete018000–020000 is read/selected. All original files remain untouched.


In [ ]:
from checkpoint_guard import sha256, verify_checkpoint

EXISTING_LOCAL_GOLD_ZIP = ""  # optional: already verified Linux ZIP in this runtime
GOLDEN_INPUT_FILE_ID = ""  # only needed if shared017000 is incomplete and no local ZIP
INPUT = Path("/content") / (DRIVE_RUN_ID + "-input")
ORIGINAL = INPUT / "training-tiago-v1"
RECOVERY = Path("/content") / (DRIVE_RUN_ID + "-recovery")
assert not INPUT.exists() and not RECOVERY.exists(), "Preserve existing attempts"
required = tuple(gold["files_sha256"])
shared17 = DRIVE_ROOT / "training-tiago-v1/checkpoints/017000"
shared_ok = False
try:
    record = verify_checkpoint(shared17, required)
    shared_ok = (
        record["files_sha256"] == gold["files_sha256"]
        and sha256(shared17 / "COMPLETE.json") == gold["source_complete_sha256"]
    )
except (OSError, ValueError, KeyError, TypeError):
    pass
saved = ORIGINAL / "checkpoints/017000"
if shared_ok:
    shutil.copytree(shared17, saved)  # source read only; hashes are pinned by Linux backup
else:
    if EXISTING_LOCAL_GOLD_ZIP:
        input_zip = Path(EXISTING_LOCAL_GOLD_ZIP)
    else:
        assert GOLDEN_INPUT_FILE_ID, (
            "STOP: provide the Linux-golden ZIP; do not use incomplete checkpoints"
        )
        input_zip = Path("/content") / (DRIVE_RUN_ID + "-golden017000.zip")
        client.download_input(GOLDEN_INPUT_FILE_ID, input_zip, gold["bytes"], gold["sha256"])
    assert input_zip.stat().st_size == gold["bytes"]
    assert sha256(input_zip) == gold["sha256"]
    with zipfile.ZipFile(input_zip) as archive:
        assert set(archive.namelist()) == set(required) | {"COMPLETE.json"}
    safe_extract(input_zip, saved)
record = verify_checkpoint(saved, required)
assert record["files_sha256"] == gold["files_sha256"]
assert sha256(saved / "COMPLETE.json") == gold["source_complete_sha256"]
assert sha256(STAGING / "source.zip") == (
    "2d3cb4bd06ac8313b1bb3cdb5aae7f916c2b614dd3884172c4570be9ff36b2f5"
)
shutil.copyfile(STAGING / "source.zip", INPUT / "source.zip")
shutil.copyfile(
    DRIVE_HELPERS / "evidence/original-optimizer-steps.jsonl", ORIGINAL / "optimizer-steps.jsonl"
)
SESSION_CONTEXT = Path("/content") / (DRIVE_RUN_ID + "-account-session.json")
session = dict(context)
session["installed_versions"] = versions
session["handoff_transport"] = "drive_api"
session["destination_folder"] = folder
session["api_canary_receipt_sha256"] = sha256(Path(config["canary_receipt"]))
with SESSION_CONTEXT.open("x", encoding="utf-8") as stream:
    json.dump(session, stream, indent=2)
COMMAND = [str(PYTHON), str(DRIVE_HELPERS / "recover_training.py")]
PATHS = [
    "--project",
    str(SOURCE),
    "--dataset",
    str(DATASET),
    "--original",
    str(ORIGINAL),
    "--recovery",
    str(RECOVERY),
    "--processor-dir",
    str(DRIVE_HELPERS / "processor"),
    "--session-context",
    str(SESSION_CONTEXT),
]
subprocess.run([*COMMAND, "audit", *PATHS], check=True)
subprocess.run([*COMMAND, "prepare", *PATHS], check=True)
print("LINUX_PINNED_017000_LOCAL_PREPARATION_VERIFIED", RECOVERY)

If source verification/copy fails, STOP and preserve the partial attempt; do not delete evidence or
substitute another checkpoint. No setup from completed Cells1–4 is rerun.

## Replacement7 — actual T4 restoration, zero optimizer updates


In [ ]:
gpu_check = """
import torch
assert torch.cuda.is_available(), 'STOP: CUDA unavailable'
assert 'T4' in torch.cuda.get_device_name(0), 'STOP: approved T4 required'
print('ACTUAL_T4_AVAILABLE', torch.cuda.get_device_name(0))
"""
subprocess.run([str(PYTHON), "-c", gpu_check], check=True)
subprocess.run([*COMMAND, "restore-check", *PATHS], check=True)
restored = json.loads((RECOVERY / "restore-check.json").read_text())
assert restored["restored_step"] == 17000 and restored["device"] == "cuda"
assert restored["optimizer_updates_at_restore_gate"] == 0
assert restored["actual_installed_smolvla"] and restored["scheduler_verified"]
assert restored["adamw_parameter_groups_verified"]
assert restored["model_tensor_values_verified"] > 0
assert restored["optimizer_tensor_values_verified"] > 0
assert restored["rng_tensors_verified"] > 0
assert restored["all_checkpoint_files_verified"] == len(required)
assert restored["relocated_processor_directory"] == str(DRIVE_HELPERS / "processor")
print("STOP_READY: model/AdamW/scheduler/RNG restored17000; zero optimizer updates")

## Replacement8 — manual future launch only (commented out)
Run only after all preceding gates pass and the owner chooses to launch. Target20000, batch8, seed0,
approved80 episodes and save1000 remain unchanged. The callback uploads full018000/019000 synchronously;
at20000 it uploads/verifies the portable inference model FIRST, then full resumable state.
Each upload requires API ownership/size/checksum verification; absent serverSHA requires streamed
API SHA-256 read-back. MD5 alone never passes. Verified receipts are uploaded beside the artifacts.


In [ ]:
# No training starts as delivered. Owner manually launches only after all gates pass:
# subprocess.run([*COMMAND, 'train', *PATHS, '--execute-recovery',
#                 '--durability-config', str(CONFIG)], check=True)

Expected future first event:17001/restored17000/calls1. Final:20000/restored17000/calls3000.
Expect DRIVE_API_ARTIFACT_VERIFIED and DRIVE_API_CHECKPOINT_VERIFIED logs with file IDs.
Keep the notebook kernel alive for token refresh. Do not copy the token file into Drive or archives.
If an API gate fails, training stops; preserve local/remote partial objects and logs. Do not blindly
restart from017000 after a durable018000/019000; the existing partial-resume limitation remains.
If final portable succeeds but full state fails, preserve/download the portable and repair upload
only—do not redo optimizer updates. No automatic evaluation or TEST use.

Remaining risk: API verification is point-in-time evidence, not proof of indefinite persistence.
The new owner's quota/policy, future deletion and Colab session expiry can still cause loss. Download
the single verified final portable ZIP to Linux immediately after completion and verify SHA there.
Large upload speed is not measured by the1MiB canary. Plan ~47minutes compute PLUS about6GB upload,
and potentially up to6GB API read-back when serverSHA is absent. Do not reduce frozen training settings.
